<a href="https://colab.research.google.com/github/milicaclm/transportni_problem/blob/main/transportni_problem_projekat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import numpy as np
import pandas as pd

In [64]:
kapacitet = [2, 6, 7]
potražnja = [3, 3, 4, 5]

cene = np.array([
    [20, 11, 15, 13],
    [17, 14, 12, 13],
    [15, 12, 18, 18]
])

cost_df = pd.DataFrame(cene, columns=['D1', 'D2', 'D3', 'D4'], index=['F1', 'F2', 'F3'])

print("Kapaciteti (Supply):", kapacitet)
print("Zahtevi (Demand):", potražnja)
print("Troškovi transporta (Costs):")
print(cost_df)

Kapaciteti (Supply): [2, 6, 7]
Zahtevi (Demand): [3, 3, 4, 5]
Troškovi transporta (Costs):
    D1  D2  D3  D4
F1  20  11  15  13
F2  17  14  12  13
F3  15  12  18  18


In [65]:
def severozapadni_ugao(kapacitet, potražnja):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    i = 0
    j = 0

    while i < len(kapacitet) and j < len(potražnja):
        if kap[i] <= pot[j]:
            resenje[i][j] = kap[i]
            pot[j] -= kap[i]
            i += 1
        else:
            resenje[i][j] = pot[j]
            kap[i] -= pot[j]
            j += 1

    return resenje

In [66]:
def metod_najmanjih_cena(kapacitet, potražnja, cene):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    cost_indices = [(cene[i][j], i, j) for i in range(len(kapacitet)) for j in range(len(potražnja))]
    cost_indices.sort()

    for cost, i, j in cost_indices:
        if kap[i] == 0 or pot[j] == 0:
            continue
        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

    return resenje

In [67]:
def fogelov_metod(kapacitet, potražnja, costs):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    INF = int(1e10)

    while np.sum(kap) > 0 and np.sum(pot) > 0:

        redovni_penali = []
        for i in range(len(kap)):
            if kap[i] == 0:
                redovni_penali.append(-INF)
            else:
                red = sorted([costs[i][j] for j in range(len(pot)) if pot[j] > 0])
                if len(red) > 1:
                    redovni_penali.append(red[1] - red[0])
                else:
                    redovni_penali.append(red[0] if len(red) > 0 else -INF)

        kolonski_penali = []
        for j in range(len(pot)):
            if pot[j] == 0:
                kolonski_penali.append(-INF)
            else:
                kolona = sorted([costs[i][j] for i in range(len(kap)) if kap[i] > 0])
                if len(kolona) > 1:
                    kolonski_penali.append(kolona[1] - kolona[0])
                else:
                    kolonski_penali.append(kolona[0] if len(kolona) > 0 else -INF)

        max_red_penal = max(redovni_penali)
        max_kol_penal = max(kolonski_penali)

        if max_red_penal > max_kol_penal:
            i = redovni_penali.index(max_red_penal)
            j = np.argmin([costs[i][j] if pot[j] > 0 else INF for j in range(len(pot))])
        else:
            j = kolonski_penali.index(max_kol_penal)
            i = np.argmin([costs[i][j] if kap[i] > 0 else INF for i in range(len(kap))])

        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

        if kap[i] == 0:
            for k in range(len(pot)):
                costs[i][k] = INF
        if pot[j] == 0:
            for k in range(len(kap)):
                costs[k][j] = INF

    return resenje

In [68]:
resenje = severozapadni_ugao(kapacitet, potražnja)
resenje_df = pd.DataFrame(resenje)

print("Inicijalno rešenje (Severozapadni ugao):")
print(resenje_df)

#resenje = metod_najmanjih_cena(kapacitet, potražnja, cene)
#resenje_df = pd.DataFrame(resenje)

#print("Inicijalno rešenje (Metod najmanjih cena):")
#print(resenje_df)

#resenje = fogelov_metod(kapacitet, potražnja, cene)
#resenje_df = pd.DataFrame(resenje, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

#print("Inicijalno rešenje (Fogelov metod):")
#print(resenje_df)

Inicijalno rešenje (Severozapadni ugao):
     0    1    2    3
0  2.0  0.0  0.0  0.0
1  1.0  3.0  2.0  0.0
2  0.0  0.0  2.0  5.0


In [ ]:
def iterativni_postupak(cene: np.ndarray, inicijalno_resenje: pd.DataFrame):
    u = {idx: None for idx in inicijalno_resenje.index}
    v = {col: None for col in inicijalno_resenje.columns}
    c_prim = np.zeros_like(cene)
    i_min = None
    j_min = None

    # Stavlja nulu u 'u' za najpopunjeniji red inicijalnog resenja (sto manje praznih polja)
    najpopunjeniji_red = inicijalno_resenje.sum(axis=1).idxmax()
    u[najpopunjeniji_red] = 0

    # c[i][j] == u[i] + v[j]
    def korak1():
        nonlocal u, v
        while True:
            promena = False
            for i_idx in inicijalno_resenje.index:
                for j_col in inicijalno_resenje.columns:
                    i = inicijalno_resenje.index.get_loc(i_idx)
                    j = inicijalno_resenje.columns.get_loc(j_col)
                    if inicijalno_resenje.loc[i_idx, j_col] > 0:
                        if u[i_idx] is not None and v[j_col] is not None:
                            if cene[i, j] != u[i_idx] + v[j_col]:
                                raise ValueError("Neispravna vrednost u[i] + v[j]")
                            continue
                        elif u[i_idx] is None and v[j_col] is not None:
                            u[i_idx] = cene[i, j] - v[j_col]
                            promena = True
                        elif v[j_col] is None and u[i_idx] is not None:
                            v[j_col] = cene[i, j] - u[i_idx]
                            promena = True
            if not promena:
                break
    def korak2():
        nonlocal c_prim, j_min, i_min
        c_prim = np.zeros_like(cene)
        for i_idx in inicijalno_resenje.index:
            for j_col in inicijalno_resenje.columns:
                i = inicijalno_resenje.index.get_loc(i_idx)
                j = inicijalno_resenje.columns.get_loc(j_col)
                if inicijalno_resenje.loc[i_idx, j_col] == 0:
                    if u[i_idx] is not None and v[j_col] is not None:
                        c_prim[i, j] = cene[i, j] - u[i_idx] - v[j_col]
                        if c_prim[i, j] < 0 and (i_min is None or j_min is None or c_prim[i, j] < c_prim[i_min, j_min]):
                            i_min = i
                            j_min = j

    def korak3():
        nonlocal i_min, j_min, inicijalno_resenje

        from collections import deque
 
        m, n = inicijalno_resenje.shape
        index_list = list(inicijalno_resenje.index)
        columns_list = list(inicijalno_resenje.columns)

        # Pronađi sve ćelije sa alokacijom > 0
        allocated = []
        for i_idx in index_list:
            for j_col in columns_list:
                if inicijalno_resenje.loc[i_idx, j_col] > 0:
                    allocated.append((i_idx, j_col))
        # Dodaj ulaznu ćeliju
        i_idx_min = index_list[i_min]
        j_col_min = columns_list[j_min]
        allocated.append((i_idx_min, j_col_min))

        # DFS za pronalaženje zatvorene petlje: naizmenično h/v pokreti
        def find_loop(path, visited, direction):
            i_idx, j_col = path[-1]
            for ii_idx, jj_col in allocated:
                if (ii_idx, jj_col) == (i_idx_min, j_col_min) and len(path) >= 4 and direction != path[-2][0] == path[-1][0]:
                    return path
                if (ii_idx, jj_col) in visited:
                    continue
                # Mora biti isti red ili ista kolona
                if direction == "row" and ii_idx == i_idx and jj_col != j_col:
                    new_path = path + [(ii_idx, jj_col)]
                    result = find_loop(new_path, visited | {(ii_idx, jj_col)}, "col")
                    if result:
                        return result
                elif direction == "col" and jj_col == j_col and ii_idx != i_idx:
                    new_path = path + [(ii_idx, jj_col)]
                    result = find_loop(new_path, visited | {(ii_idx, jj_col)}, "row")
                    if result:
                        return result
            return None

        loop = find_loop([(i_idx_min, j_col_min)], {(i_idx_min, j_col_min)}, "row") or \
               find_loop([(i_idx_min, j_col_min)], {(i_idx_min, j_col_min)}, "col")

        if loop is None:
            raise Exception("Nije moguće formirati zatvorenu petlju")

        # Označi ćelije kao + i - naizmenično, početna je +
        plus_minus = [1 if idx % 2 == 0 else -1 for idx in range(len(loop))]

        # Nađi minimalnu količinu u ćelijama sa minusom
        minus_values = [inicijalno_resenje.loc[i_idx, j_col] for (sign, (i_idx, j_col)) in zip(plus_minus, loop) if sign == -1]
        theta = min(minus_values)

        # Ažuriraj alokaciju
        for (sign, (i_idx, j_col)) in zip(plus_minus, loop):
            inicijalno_resenje.loc[i_idx, j_col] += sign * theta

        # Ažuriraj alokaciju
        for (sign, (i, j)) in zip(plus_minus, loop):
            inicijalno_resenje.iloc[i, j] += sign * theta

    korak1()
    korak2()
    # Ako nema negativnih c_prim, rešenje je optimalno
    if i_min is None or j_min is None:
        return inicijalno_resenje
    korak3()
    return inicijalno_resenje

iterativni_postupak(cene, resenje_df)

,0,1,2,3
0,2.0,0.0,0.0,0.0
1,-1.0,3.0,4.0,0.0
2,2.0,0.0,0.0,5.0
